In [ ]:
import numpy as np
import pandas as pd
import warnings
import gc
import time
from itertools import combinations

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize, Bounds
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Libraries loaded!')

In [ ]:
class CFG:
    TARGET        = 'Churn'
    N_FOLDS       = 20
    INNER_FOLDS   = 5
    RANDOM_SEED   = 42

    TRAIN_PATH    = r'C:\Users\tamkn\Downloads\train.csv'
    TEST_PATH     = r'C:\Users\tamkn\Downloads\test.csv'
    ORIGINAL_PATH = r'C:\Users\tamkn\Downloads\WA_Fn-UseC_-Telco-Customer-Churn.csv'

XGB_PARAMS = {
    'n_estimators'         : 50000,
    'learning_rate'        : 0.0063,
    'max_depth'            : 5,
    'subsample'            : 0.81,
    'colsample_bytree'     : 0.32,
    'min_child_weight'     : 6,
    'reg_alpha'            : 3.5017,
    'reg_lambda'           : 1.2925,
    'gamma'                : 0.790,
    'random_state'         : CFG.RANDOM_SEED,
    'early_stopping_rounds': 500,
    'objective'            : 'binary:logistic',
    'eval_metric'          : 'auc',
    'enable_categorical'   : True,
    'device'               : 'cuda',
    'verbosity'            : 0,
}

CB_PARAMS = {
    'n_estimators'         : 50000,
    'learning_rate'        : 0.05,
    'eval_metric'          : 'AUC',
    'max_depth'            : 5,
    'auto_class_weights'   : 'Balanced',
    'random_state'         : CFG.RANDOM_SEED,
    'early_stopping_rounds': 100,
    'task_type'            : 'GPU',
    'verbose'              : False,
}

models = {
    'XGB': XGBClassifier(**XGB_PARAMS),
    'CB' : CatBoostClassifier(**CB_PARAMS),
}
print('Models: XGB + CB')

## 1. データ読み込み

In [ ]:
print('Loading datasets...')
train = pd.read_csv(CFG.TRAIN_PATH)
test  = pd.read_csv(CFG.TEST_PATH)
orig  = pd.read_csv(CFG.ORIGINAL_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map({'No': 0, 'Yes': 1}).astype(int)
orig[CFG.TARGET]  = orig[CFG.TARGET].map({'No': 0, 'Yes': 1}).astype(int)

orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
orig['TotalCharges'].fillna(orig['TotalCharges'].median(), inplace=True)

if 'customerID' in orig.columns:
    orig.drop(columns=['customerID'], inplace=True)

train_ids = train['id'].copy()
test_ids  = test['id'].copy()

# origをtrainにconcat
orig_for_train = orig.copy()
orig_for_train['id'] = -1
train = pd.concat([train, orig_for_train], ignore_index=True)

print(f'Train : {train.shape}  (orig concatted)')
print(f'Test  : {test.shape}')
print(f'Orig  : {orig.shape}')
print(f'Churn rate: {train[CFG.TARGET].mean()*100:.2f}%')

## 2. EDA（簡易）

In [ ]:
display(train.head(3))
print(train.dtypes.to_frame('dtype'))
print('\nMissing:')
print(train.isnull().sum()[train.isnull().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#2ecc71', '#e74c3c']
train_churn = train[CFG.TARGET].value_counts()
orig_churn  = orig[CFG.TARGET].value_counts()

for ax, churn, title in zip(axes[:2],
                             [train_churn, orig_churn],
                             ['Training Set', 'Original Dataset']):
    ax.pie(churn.values, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
           colors=colors, explode=[0, 0.05], startangle=90)
    ax.set_title(f'{title} - Churn Distribution')

x, width = np.arange(2), 0.35
axes[2].bar(x - width/2, [train_churn[0], orig_churn[0]], width, label='No Churn', color='#2ecc71')
axes[2].bar(x + width/2, [train_churn[1], orig_churn[1]], width, label='Churn',    color='#e74c3c')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Training', 'Original'])
axes[2].set_title('Target Distribution Comparison')
axes[2].legend()
plt.tight_layout()
plt.show()
print(f'Churn Rate — Train: {train[CFG.TARGET].mean()*100:.2f}%  /  Orig: {orig[CFG.TARGET].mean()*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges']):
    for val, label, color in [(0, 'No Churn', '#2ecc71'), (1, 'Churn', '#e74c3c')]:
        axes[idx].hist(train[train[CFG.TARGET]==val][col],
                       bins=30, alpha=0.6, label=label, color=color, density=True)
    axes[idx].set(xlabel=col, ylabel='Density', title=f'{col} by Churn')
    axes[idx].legend()
plt.tight_layout()
plt.show()

In [ ]:
CATS_PLOT = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()
for idx, col in enumerate(CATS_PLOT):
    cr = train.groupby(col)[CFG.TARGET].mean().sort_values(ascending=False)
    clrs = plt.cm.RdYlGn_r(np.linspace(0, 1, len(cr)))
    bars = axes[idx].bar(range(len(cr)), cr.values, color=clrs)
    axes[idx].set_xticks(range(len(cr)))
    axes[idx].set_xticklabels(cr.index, rotation=45, ha='right', fontsize=8)
    axes[idx].set(ylabel='Churn Rate', title=col)
    axes[idx].axhline(y=train[CFG.TARGET].mean(), color='black', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, cr.values):
        axes[idx].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                       f'{val:.2f}', ha='center', va='bottom', fontsize=7)
plt.suptitle('Churn Rate by Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
sns.heatmap(train[num_cols + [CFG.TARGET]].corr(), annot=True, cmap='RdYlGn',
            center=0, ax=axes[0], fmt='.3f', square=True)
axes[0].set_title('Numerical Features Correlation')
pivot = train.pivot_table(values=CFG.TARGET, index='Contract',
                           columns='InternetService', aggfunc='mean')
sns.heatmap(pivot, annot=True, cmap='RdYlGn_r', center=0.26,
            ax=axes[1], fmt='.3f', square=True)
axes[1].set_title('Churn Rate: Contract x InternetService')
plt.tight_layout()
plt.show()

## 3. 特徴量エンジニアリング

In [ ]:
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
NUMS       = ['tenure', 'MonthlyCharges', 'TotalCharges']
NEW_NUMS   = []
NUM_AS_CAT = []
print('Feature Engineering Pipeline Started...')

In [ ]:
# [1/7] Frequency Encoding
for col in NUMS:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{col}')
print(f'[1/7] Frequency Encoding done ({len(NUMS)} features)')

In [ ]:
# [2/7] Arithmetic Interactions
for df in [train, test, orig]:
    df['charges_deviation']      = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    df['avg_monthly_charges']    = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')
NEW_NUMS += ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges']
print('[2/7] Arithmetic Interactions done (3 features)')

In [ ]:
# [3/7] Service Counts
SERVICE_COLS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for df in [train, test, orig]:
    df['service_count'] = (df[SERVICE_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['has_internet']  = (df['InternetService'] != 'No').astype('float32')
    df['has_phone']     = (df['PhoneService'] == 'Yes').astype('float32')
NEW_NUMS += ['service_count', 'has_internet', 'has_phone']
print('[3/7] Service Counts done (3 features)')

In [ ]:
# [4/7] ORIG_proba (top 3 cols)
ORIG_PROBA_COLS = ['Contract', 'InternetService', 'tenure']
for col in ORIG_PROBA_COLS:
    tmp   = orig.groupby(col)[CFG.TARGET].mean()
    _name = f'ORIG_proba_{col}'
    train = train.merge(tmp.rename(_name), on=col, how='left')
    test  = test.merge(tmp.rename(_name),  on=col, how='left')
    for df in [train, test]:
        df[_name] = df[_name].fillna(0.5).astype('float32')
    NEW_NUMS.append(_name)
print(f'[4/7] ORIG_proba done ({len(ORIG_PROBA_COLS)} features)')

In [ ]:
# [5/7] Distribution Features
def pctrank_against(values, reference):
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    mu, sigma = np.mean(reference), np.std(reference)
    return (np.zeros(len(values), dtype='float32') if sigma == 0
            else ((values - mu) / sigma).astype('float32'))

orig_churner_tc    = orig.loc[orig[CFG.TARGET] == 1, 'TotalCharges'].values
orig_nonchurner_tc = orig.loc[orig[CFG.TARGET] == 0, 'TotalCharges'].values
orig_tc            = orig['TotalCharges'].values
orig_is_mc_mean    = orig.groupby('InternetService')['MonthlyCharges'].mean()

for df in [train, test]:
    tc = df['TotalCharges'].values
    df['pctrank_nonchurner_TC'] = pctrank_against(tc, orig_nonchurner_tc)
    df['pctrank_churner_TC']    = pctrank_against(tc, orig_churner_tc)
    df['pctrank_orig_TC']       = pctrank_against(tc, orig_tc)
    df['zscore_churn_gap_TC']   = (np.abs(zscore_against(tc, orig_churner_tc)) -
                                   np.abs(zscore_against(tc, orig_nonchurner_tc))).astype('float32')
    df['zscore_nonchurner_TC']  = zscore_against(tc, orig_nonchurner_tc)
    df['pctrank_churn_gap_TC']  = (pctrank_against(tc, orig_churner_tc) -
                                   pctrank_against(tc, orig_nonchurner_tc)).astype('float32')
    df['resid_IS_MC'] = (df['MonthlyCharges'] - df['InternetService'].map(orig_is_mc_mean).fillna(0)).astype('float32')
    for col_cat, feat_name in [('InternetService', 'cond_pctrank_IS_TC'),
                                ('Contract',        'cond_pctrank_C_TC')]:
        vals = np.zeros(len(df), dtype='float32')
        for cat_val in orig[col_cat].unique():
            mask = df[col_cat] == cat_val
            ref  = orig.loc[orig[col_cat] == cat_val, 'TotalCharges'].values
            if len(ref) > 0 and mask.sum() > 0:
                vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
        df[feat_name] = vals

DIST_FEATURES = [
    'pctrank_nonchurner_TC', 'zscore_churn_gap_TC', 'pctrank_churn_gap_TC',
    'resid_IS_MC', 'cond_pctrank_IS_TC', 'zscore_nonchurner_TC',
    'pctrank_orig_TC', 'pctrank_churner_TC', 'cond_pctrank_C_TC'
]
NEW_NUMS += DIST_FEATURES
print(f'[5/7] Distribution Features done ({len(DIST_FEATURES)} features)')

In [ ]:
# [6/7] Quantile Distance Features
for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
    ch_q = np.quantile(orig_churner_tc, q_val)
    nc_q = np.quantile(orig_nonchurner_tc, q_val)
    for df in [train, test]:
        df[f'dist_To_ch_{q_label}']   = np.abs(df['TotalCharges'] - ch_q).astype('float32')
        df[f'dist_To_nc_{q_label}']   = np.abs(df['TotalCharges'] - nc_q).astype('float32')
        df[f'qdist_gap_To_{q_label}'] = (df[f'dist_To_nc_{q_label}'] -
                                          df[f'dist_To_ch_{q_label}']).astype('float32')
QDIST_FEATURES = [
    'qdist_gap_To_q50', 'dist_To_ch_q50', 'dist_To_nc_q50',
    'dist_To_nc_q25',   'qdist_gap_To_q25',
    'dist_To_nc_q75',   'dist_To_ch_q75', 'qdist_gap_To_q75'
]
NEW_NUMS += QDIST_FEATURES
print(f'[6/7] Quantile Distance done ({len(QDIST_FEATURES)} features)')

In [ ]:
# [7/7] Numericals as Categories
for col in NUMS:
    _new = f'CAT_{col}'
    NUM_AS_CAT.append(_new)
    for df in [train, test]:
        df[_new] = df[col].astype(str).astype('category')
print(f'[7/7] Num-as-Cat done ({len(NUMS)} features)')

In [ ]:
# Digit Features
DIGIT_FEATURES = [
    'tenure_years', 'tenure_num_digits', 'tenure_rounded_10',
    'mc_fractional', 'mc_dev_from_round10', 'mc_per_digit',
    'tc_mod100', 'tc_fractional', 'tc_dev_from_round100',
]
for df in [train, test]:
    df['tenure_years']         = df['tenure'] // 12
    df['tenure_num_digits']    = df['tenure'].astype(str).str.len()
    df['tenure_rounded_10']    = np.round(df['tenure'] / 10) * 10
    df['mc_fractional']        = df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])
    mc_rounded                 = np.round(df['MonthlyCharges'] / 10) * 10
    df['mc_dev_from_round10']  = np.abs(df['MonthlyCharges'] - mc_rounded)
    mc_nd                      = np.floor(df['MonthlyCharges']).astype(int).astype(str).str.len()
    df['mc_per_digit']         = df['MonthlyCharges'] / (mc_nd.astype(float) + 0.001)
    df['tc_mod100']            = np.floor(df['TotalCharges']) % 100
    df['tc_fractional']        = df['TotalCharges'] - np.floor(df['TotalCharges'])
    tc_rounded                 = np.round(df['TotalCharges'] / 100) * 100
    df['tc_dev_from_round100'] = np.abs(df['TotalCharges'] - tc_rounded)
NEW_NUMS += DIGIT_FEATURES
print(f'Digit Features done ({len(DIGIT_FEATURES)} features)')

In [ ]:
# N-gram Features
KEEP_NGRAMS = [
    ('BG', ['Contract', 'InternetService']),
    ('TG', ['Contract', 'InternetService', 'OnlineSecurity']),
    ('TG', ['Contract', 'InternetService', 'PaymentMethod']),
    ('BG', ['Contract', 'OnlineSecurity']),
    ('TG', ['Contract', 'PaymentMethod', 'OnlineSecurity']),
    ('BG', ['Contract', 'PaymentMethod']),
    ('TG', ['InternetService', 'PaymentMethod', 'OnlineSecurity']),
    ('BG', ['PaymentMethod', 'PaperlessBilling']),
    ('BG', ['TechSupport', 'PaperlessBilling']),
    ('BG', ['PaymentMethod', 'TechSupport']),
    ('BG', ['Contract', 'TechSupport']),
    ('BG', ['PaymentMethod', 'OnlineSecurity']),
]
BIGRAM_COLS = []; TRIGRAM_COLS = []
for prefix, cat_cols in KEEP_NGRAMS:
    col_name = prefix + '_' + '_'.join(cat_cols)
    for df in [train, test]:
        df[col_name] = df[cat_cols[0]].astype(str)
        for c in cat_cols[1:]:
            df[col_name] = df[col_name] + '_' + df[c].astype(str)
        df[col_name] = df[col_name].astype('category')
    (BIGRAM_COLS if prefix == 'BG' else TRIGRAM_COLS).append(col_name)
NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS
print(f'N-gram done ({len(NGRAM_COLS)} features)')

In [ ]:
# EDA-derived Features
contract_risk = {'Month-to-month': 1.0, 'One year': 0.26, 'Two year': 0.07}
for df in [train, test, orig]:
    df['is_vulnerable'] = (
        (df['OnlineSecurity'] == 'No') &
        (df['TechSupport']    == 'No') &
        (df['InternetService']== 'Fiber optic')
    ).astype('float32')
    df['is_early_tenure']         = (df['tenure'] <= 6).astype('float32')
    df['is_electronic_check']     = (df['PaymentMethod'] == 'Electronic check').astype('float32')
    df['contract_risk_score']     = (df['Contract'].map(contract_risk).fillna(0.5) * df['MonthlyCharges']).astype('float32')
    df['tenure_charge_stability'] = (df['tenure'] / (df['MonthlyCharges'] + 1)).astype('float32')
NEW_NUMS += ['is_vulnerable', 'is_early_tenure', 'is_electronic_check',
             'contract_risk_score', 'tenure_charge_stability']
print('EDA features done (5 features)')

FEATURES         = NUMS + CATS + NEW_NUMS + NUM_AS_CAT + NGRAM_COLS
TE_COLUMNS       = NUM_AS_CAT + CATS
TE_NGRAM_COLUMNS = NGRAM_COLS
TO_REMOVE        = NUM_AS_CAT + CATS + NGRAM_COLS
STATS            = ['std', 'min', 'max']
print(f'\nTotal Features: {len(FEATURES)}')

## 4. 学習ループ（20-fold CV）

In [ ]:
print('=' * 60)
print(f'TRAINING WITH {CFG.N_FOLDS}-FOLD CROSS-VALIDATION')
print('=' * 60)

np.random.seed(CFG.RANDOM_SEED)
skf_outer = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
skf_inner = StratifiedKFold(n_splits=CFG.INNER_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)

oof         = {m: np.zeros(len(train)) for m in models}
pred        = {m: np.zeros(len(test))  for m in models}
fold_scores = {m: []                   for m in models}
fi          = {m: pd.DataFrame()       for m in models}
t0 = time.time()

for i, (train_idx, val_idx) in enumerate(skf_outer.split(train, train[CFG.TARGET])):
    print(f'\n{"="*50}')
    print(f'Fold {i+1}/{CFG.N_FOLDS}')
    print(f'{"="*50}')

    X_tr  = train.loc[train_idx, FEATURES + [CFG.TARGET]].reset_index(drop=True).copy()
    y_tr  = train.loc[train_idx, CFG.TARGET].values
    X_val = train.loc[val_idx,   FEATURES].reset_index(drop=True).copy()
    y_val = train.loc[val_idx,   CFG.TARGET].values
    X_te  = test[FEATURES].reset_index(drop=True).copy()

    # ── Inner KFold TE (categorical) ─────────────────────────────────────────
    for j, (in_tr, in_va) in enumerate(skf_inner.split(X_tr, y_tr)):
        X_tr2 = X_tr.loc[in_tr, FEATURES + [CFG.TARGET]].copy()
        X_va2 = X_tr.loc[in_va, FEATURES].copy()
        for col in TE_COLUMNS:
            tmp = X_tr2.groupby(col, observed=False)[CFG.TARGET].agg(STATS)
            tmp.columns = [f'TE1_{col}_{s}' for s in STATS]
            X_va2 = X_va2.merge(tmp, on=col, how='left')
            for c in tmp.columns:
                X_tr.loc[in_va, c] = X_va2[c].values.astype('float32')

    for col in TE_COLUMNS:
        tmp = X_tr.groupby(col, observed=False)[CFG.TARGET].agg(STATS)
        tmp.columns = [f'TE1_{col}_{s}' for s in STATS]
        tmp = tmp.astype('float32')
        X_val = X_val.merge(tmp, on=col, how='left')
        X_te  = X_te.merge(tmp, on=col, how='left')
        for c in tmp.columns:
            for df in [X_tr, X_val, X_te]:
                df[c] = df[c].fillna(0)

    # ── Inner KFold TE (N-gram) ───────────────────────────────────────────────
    for j, (in_tr, in_va) in enumerate(skf_inner.split(X_tr, y_tr)):
        X_tr2 = X_tr.loc[in_tr].copy()
        for col in TE_NGRAM_COLUMNS:
            ng_te   = X_tr2.groupby(col, observed=False)[CFG.TARGET].mean()
            ng_name = f'TE_ng_{col}'
            mapped  = X_tr.loc[in_va, col].astype(str).map(ng_te)
            X_tr.loc[in_va, ng_name] = pd.to_numeric(mapped, errors='coerce').fillna(0.5).astype('float32').values

    for col in TE_NGRAM_COLUMNS:
        ng_te   = X_tr.groupby(col, observed=False)[CFG.TARGET].mean()
        ng_name = f'TE_ng_{col}'
        X_val[ng_name] = pd.to_numeric(X_val[col].astype(str).map(ng_te), errors='coerce').fillna(0.5).astype('float32')
        X_te[ng_name]  = pd.to_numeric(X_te[col].astype(str).map(ng_te),  errors='coerce').fillna(0.5).astype('float32')
        if ng_name not in X_tr.columns:
            X_tr[ng_name] = 0.5
        else:
            X_tr[ng_name] = pd.to_numeric(X_tr[ng_name], errors='coerce').fillna(0.5).astype('float32')

    # ── sklearn TargetEncoder (mean) ─────────────────────────────────────────
    TE_MEAN_COLS = [f'TE_{col}' for col in TE_COLUMNS]
    te = TargetEncoder(cv=CFG.INNER_FOLDS, shuffle=True, smooth='auto',
                       target_type='binary', random_state=CFG.RANDOM_SEED)
    X_tr[TE_MEAN_COLS]  = te.fit_transform(X_tr[TE_COLUMNS], y_tr)
    X_val[TE_MEAN_COLS] = te.transform(X_val[TE_COLUMNS])
    X_te[TE_MEAN_COLS]  = te.transform(X_te[TE_COLUMNS])

    # ── カテゴリ削除・整形 ────────────────────────────────────────────────────
    for df in [X_tr, X_val, X_te]:
        for c in CATS + NUM_AS_CAT:
            if c in df.columns:
                df[c] = df[c].astype(str).astype('category')
        df.drop(columns=[c for c in TO_REMOVE if c in df.columns], inplace=True, errors='ignore')
    X_tr.drop(columns=[CFG.TARGET], inplace=True, errors='ignore')
    COLS_FIT = X_tr.columns

    # ── 各モデル学習 ─────────────────────────────────────────────────────────
    for model_name, model in models.items():
        if model_name == 'CB':
            cat_cols = [c for c in X_tr.columns if X_tr[c].dtype.name in ['category', 'object']]
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_cols, verbose=False)
        else:  # XGB
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=1000)

        fold_imp = pd.DataFrame({'feature': COLS_FIT,
                                  f'importance_fold_{i+1}': model.feature_importances_})
        fi[model_name] = pd.merge(fi[model_name], fold_imp, on='feature', how='outer') if i > 0 else fold_imp

        oof[model_name][val_idx] = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof[model_name][val_idx])
        fold_scores[model_name].append(fold_auc)
        pred[model_name] += model.predict_proba(X_te[COLS_FIT])[:, 1] / CFG.N_FOLDS
        print(f'   Fold {i+1} {model_name} AUC: {fold_auc:.5f} | {(time.time()-t0)/60:.1f} min')
        del model

    gc.collect()

print('\nTRAINING COMPLETE!')
for m in models:
    print(f'[{m}] Mean CV AUC: {np.mean(fold_scores[m]):.5f} +/- {np.std(fold_scores[m]):.5f}')

## 5. アンサンブル（LRスタッキング + COBYLAブレンド）

In [ ]:
# LR スタッキング
meta_train = np.column_stack([oof[k]  for k in models])
meta_test  = np.column_stack([pred[k] for k in models])
scaler         = StandardScaler()
meta_tr_scaled = scaler.fit_transform(meta_train)
meta_te_scaled = scaler.transform(meta_test)

skf_stack  = StratifiedKFold(n_splits=10, shuffle=True, random_state=CFG.RANDOM_SEED)
oof_stack  = np.zeros(len(train))
pred_stack = np.zeros(len(test))
for tr_idx, val_idx in skf_stack.split(meta_tr_scaled, train[CFG.TARGET]):
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=CFG.RANDOM_SEED)
    lr.fit(meta_tr_scaled[tr_idx], train[CFG.TARGET].values[tr_idx])
    oof_stack[val_idx]  = lr.predict_proba(meta_tr_scaled[val_idx])[:, 1]
    pred_stack         += lr.predict_proba(meta_te_scaled)[:, 1] / skf_stack.n_splits
print(f'Stacking AUC: {roc_auc_score(train[CFG.TARGET], oof_stack):.5f}')

# COBYLA ブレンド
blend_oofs  = {**{k: oof[k]  for k in models}, 'Stack': oof_stack}
blend_preds = {**{k: pred[k] for k in models}, 'Stack': pred_stack}
blend_keys  = list(blend_oofs.keys())

def objective(w):
    ens = sum(blend_oofs[k] * w[i] for i, k in enumerate(blend_keys))
    return -roc_auc_score(train[CFG.TARGET], ens)

w0 = np.array([0.5, 0.5, 0.0])
w  = minimize(objective, w0, method='COBYLA',
              bounds=Bounds(0, 1),
              constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]).x

print('\nBlend weights:')
for k, ww in zip(blend_keys, w):
    print(f'  {k:<8}: {ww:.4f}')

oof_ensemble  = sum(blend_oofs[k]  * w[i] for i, k in enumerate(blend_keys))
pred_ensemble = sum(blend_preds[k] * w[i] for i, k in enumerate(blend_keys))
print(f'\nFinal Ensemble AUC: {roc_auc_score(train[CFG.TARGET], oof_ensemble):.5f}')

## 6. 評価・可視化

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve

print(f'Ensemble CV AUC: {roc_auc_score(train[CFG.TARGET], oof_ensemble):.5f}')
for m in models:
    ms  = np.mean(fold_scores[m])
    std = np.std(fold_scores[m])
    print(f'[{m}] OOF AUC: {roc_auc_score(train[CFG.TARGET], oof[m]):.5f}  '
          f'Mean fold: {ms:.5f} +/- {std:.5f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC
fpr, tpr, _ = roc_curve(train[CFG.TARGET], oof_ensemble)
axes[0].plot(fpr, tpr, color='#3498db', lw=2, label=f'AUC={auc(fpr,tpr):.5f}')
axes[0].plot([0,1],[0,1],'--', color='gray')
axes[0].fill_between(fpr, tpr, alpha=0.2, color='#3498db')
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curve')
axes[0].legend(); axes[0].grid(alpha=0.3)

# OOF 分布
axes[1].hist(oof_ensemble[train[CFG.TARGET]==0], bins=50, alpha=0.7,
             label='No Churn', color='#2ecc71', density=True)
axes[1].hist(oof_ensemble[train[CFG.TARGET]==1], bins=50, alpha=0.7,
             label='Churn',    color='#e74c3c', density=True)
axes[1].axvline(0.5, color='black', linestyle='--')
axes[1].legend()
axes[1].set(xlabel='Predicted Prob', title='OOF Prediction Distribution')
axes[1].grid(alpha=0.3)

# PR Curve
prec, rec, _ = precision_recall_curve(train[CFG.TARGET], oof_ensemble)
axes[2].plot(rec, prec, color='#9b59b6', lw=2)
axes[2].fill_between(rec, prec, alpha=0.2, color='#9b59b6')
axes[2].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[2].grid(alpha=0.3)

plt.suptitle('Model Evaluation (v2)', fontsize=14)
plt.tight_layout()
plt.savefig('model_evaluation_v2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 特徴量重要度
fig, axes = plt.subplots(1, len(models), figsize=(14, 8))
for ax, m in zip(axes, models):
    imp_cols = [c for c in fi[m].columns if c.startswith('importance_')]
    fi[m]['mean_importance'] = fi[m][imp_cols].mean(axis=1)
    fi_sorted = fi[m].sort_values('mean_importance', ascending=True).tail(20)
    ax.barh(fi_sorted['feature'], fi_sorted['mean_importance'],
            color='#3498db', edgecolor='white')
    ax.set_title(f'{m} Feature Importance (Top 20)')
    ax.set_xlabel('Mean Importance')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 提出ファイル保存

In [ ]:
# OOF 保存
oof_df = pd.DataFrame({'id': train_ids, CFG.TARGET: oof_ensemble})
for m in models:
    oof_df[f'{CFG.TARGET}_{m}'] = oof[m]
oof_df['Churn_Stack'] = oof_stack
oof_df.to_csv('oof_predictions_v2.csv', index=False)
print('Saved: oof_predictions_v2.csv')

# submission 保存
sub_df = pd.DataFrame({'id': test_ids, CFG.TARGET: pred_ensemble})
sub_df.to_csv('submission_v2.csv', index=False)
print('Saved: submission_v2.csv')
display(sub_df.head())
print(f'\nsubmission shape: {sub_df.shape}')
print(f'pred range: [{pred_ensemble.min():.4f}, {pred_ensemble.max():.4f}]')